# CASC - Cardiac Automated Segmentation Comparison Pipeline

This notebook provides an interactive interface to run the CASC pipeline for cardiac MRI segmentation.

## Supported Models
- **CineMA**: Convolutional Vision Transformer
- **nnFormer**: 3D Medical Image Segmentation Transformer
- **VSA-3L**: MONAI Ventricular Short Axis 3-Label

## Output Labels
- 0: Background
- 1: Right Ventricle (RV)
- 2: Myocardium (MYO)
- 3: Left Ventricle (LV)

---
## 1. Setup and Configuration

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import SimpleITK as sitk
except ImportError:
    print('SimpleITK not installed. Run: pip install SimpleITK')
from IPython.display import display, HTML, Markdown

# Configuration
CASC_ROOT = Path('/scratch/st-zlaksman-1/pmoheban/CASC')
ACDC_ROOT = Path('/scratch/st-zlaksman-1/pmoheban/ACDC/database')
DEFAULT_OUTPUT = CASC_ROOT / 'results'

sys.path.insert(0, str(CASC_ROOT / 'bin'))

print(f'CASC Root: {CASC_ROOT}')
print(f'ACDC Database: {ACDC_ROOT}')
print(f'Default Output: {DEFAULT_OUTPUT}')

---
## 2. Pipeline Configuration

In [ ]:
# PIPELINE CONFIGURATION - Modify as needed

# Input data
INPUT_SAMPLESHEET = CASC_ROOT / 'data' / 'acdc_testing_samplesheet.csv'

# Output directory
OUTPUT_DIR = DEFAULT_OUTPUT / f'run_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

# Models to run: 'cinema', 'nnformer', 'vsa3l', or 'all'
MODELS = 'all'

# Enable comparison report
COMPARE = True

# Profile: 'docker', 'singularity', 'slurm'
PROFILE = 'singularity'

# Resources
MAX_CPUS = 8
MAX_MEMORY = '32.GB'

print('Pipeline Configuration:')
print(f'  Input: {INPUT_SAMPLESHEET}')
print(f'  Output: {OUTPUT_DIR}')
print(f'  Models: {MODELS}')
print(f'  Profile: {PROFILE}')

---
## 3. View Input Data

In [ ]:
# Load and display samplesheet
samplesheet = pd.read_csv(INPUT_SAMPLESHEET)
print(f'Found {len(samplesheet)} patients in samplesheet')
display(samplesheet.head(10))

In [ ]:
# Visualize a sample patient
def visualize_patient(patient_row, slice_idx=None):
    image_path = patient_row['image']
    patient_id = patient_row['patient_id']
    
    img = sitk.ReadImage(image_path)
    data = sitk.GetArrayFromImage(img)
    
    if slice_idx is None:
        slice_idx = data.shape[1] // 2
    
    ed_frame = 0
    es_frame = data.shape[0] - 1
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(data[ed_frame, slice_idx], cmap='gray')
    axes[0].set_title(f'{patient_id} - ED (frame {ed_frame})')
    axes[0].axis('off')
    
    axes[1].imshow(data[es_frame, slice_idx], cmap='gray')
    axes[1].set_title(f'{patient_id} - ES (frame {es_frame})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f'Image shape: {data.shape}')

# Visualize first patient
visualize_patient(samplesheet.iloc[0])

---
## 4. Run Pipeline

In [ ]:
def run_casc_pipeline(input_samplesheet, output_dir, models='all', compare=True,
                        profile='singularity', max_cpus=None, max_memory=None,
                        resume=False, dry_run=False):
    cmd = [
        'nextflow', 'run', str(CASC_ROOT / 'main.nf'),
        '--input', str(input_samplesheet),
        '--outdir', str(output_dir),
        '--models', models,
        '--compare', str(compare).lower(),
        '-profile', profile,
        '-work-dir', str(output_dir / 'work')
    ]
    
    if max_cpus:
        cmd.extend(['--max_cpus', str(max_cpus)])
    if max_memory:
        cmd.extend(['--max_memory', str(max_memory)])
    if resume:
        cmd.append('-resume')
    
    print('Nextflow Command:')
    print(' '.join(cmd))
    
    if dry_run:
        print('[DRY RUN] Command not executed')
        return None
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    print('Starting pipeline...')
    
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, cwd=str(CASC_ROOT))
    
    for line in process.stdout:
        print(line, end='')
    
    process.wait()
    
    if process.returncode == 0:
        print(f'Pipeline completed! Results: {output_dir}')
    else:
        print(f'Pipeline failed with code {process.returncode}')
    
    return process.returncode

In [ ]:
# Preview command (dry run)
run_casc_pipeline(
    input_samplesheet=INPUT_SAMPLESHEET,
    output_dir=OUTPUT_DIR,
    models=MODELS,
    compare=COMPARE,
    profile=PROFILE,
    max_cpus=MAX_CPUS,
    max_memory=MAX_MEMORY,
    dry_run=True
)

In [ ]:
# UNCOMMENT TO RUN THE PIPELINE
# run_casc_pipeline(
#     input_samplesheet=INPUT_SAMPLESHEET,
#     output_dir=OUTPUT_DIR,
#     models=MODELS,
#     compare=COMPARE,
#     profile=PROFILE,
#     max_cpus=MAX_CPUS,
#     max_memory=MAX_MEMORY,
#     dry_run=False
# )

---
## 5. View Results

In [ ]:
def load_results(results_dir):
    results_dir = Path(results_dir)
    
    metrics_file = results_dir / 'comparison' / 'aggregated_metrics.csv'
    comparison_file = results_dir / 'comparison' / 'model_comparison.csv'
    
    results = {}
    
    if metrics_file.exists():
        results['metrics'] = pd.read_csv(metrics_file)
        print(f'Loaded {len(results["metrics"])} metric records')
    
    if comparison_file.exists():
        results['comparison'] = pd.read_csv(comparison_file)
        print('Loaded model comparison summary')
    
    return results

# Load results
try:
    results = load_results(DEFAULT_OUTPUT)
    if 'comparison' in results:
        display(Markdown('### Model Comparison Summary'))
        display(results['comparison'])
except Exception as e:
    print(f'Could not load results: {e}')
    print('Run the pipeline first to generate results.')

---
## 6. Docker Commands Reference

In [ ]:
docker_commands = '''
# Pull Docker images
docker pull ghcr.io/pmoheban/casc-cinema:latest
docker pull ghcr.io/pmoheban/casc-nnformer:latest
docker pull ghcr.io/pmoheban/casc-vsa3l:latest

# Run interactively
docker run --gpus all -it \
    -v /path/to/data:/data \
    -v /path/to/output:/output \
    ghcr.io/pmoheban/casc-cinema:latest bash

# Convert to Singularity
singularity pull casc-cinema.sif docker://ghcr.io/pmoheban/casc-cinema:latest
'''
print(docker_commands)

---
## 7. Troubleshooting

In [ ]:
# Check installations
!which nextflow && nextflow -version || echo 'Nextflow not found'
!which singularity && singularity --version || echo 'Singularity not installed'

In [ ]:
# Check GPU
try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch not installed')